# Phase 6 — Configuration and Secrets

**What you will learn**
- ConfigMaps: store non-secret configuration
- Secrets: store passwords and keys (and why **base64 is not encryption**)
- Environment variables: `env`, `envFrom`, `valueFrom`, Downward API
- Mounting configuration as files, and the `subPath` gotcha (no auto-update)
- External Secrets Operator (ESO) with a fake provider (runs locally) and OCI Vault (production example)
- Secret encryption at rest in etcd (EncryptionConfiguration, KMS, OKE + OCI Vault)
- Configuration management patterns: immutable ConfigMaps, hash-suffix rollouts with kustomize
- External config providers: Secrets Store CSI Driver

**Time:** about 3–4 hours.

**Prerequisites**
- Notebook 00: kind cluster `learn` running. `helm` installed.
- Notebooks 03 (workloads) and 05 (storage, volumes).

**How to use this notebook**
- Run cells top to bottom with **Shift+Enter**.
- `%%writefile` cells create YAML in `labs/06/`. Read the comments: they explain each field.
- Answers are hidden in **Answer** sections. Try first.

> All passwords in this notebook are **fake lab values**. Never put real secrets in notebooks, Git, or shell history.

## Setup: safety check and namespace

In [ ]:
%%bash
ctx=$(kubectl config current-context 2>/dev/null)
if [ "$ctx" != "kind-learn" ]; then
  echo "STOP: current context is '$ctx', not 'kind-learn'."
  echo "Fix:  kubectl config use-context kind-learn"
  exit 1
fi
echo "OK: using $ctx"

In [ ]:
%%bash
mkdir -p labs/06
kubectl create namespace lab-06 --dry-run=client -o yaml | kubectl apply -f -
kubectl config set-context --current --namespace=lab-06
echo "Default namespace is now lab-06"

---
# Topic 1 — ConfigMaps

### What is it?
A **ConfigMap** is a Kubernetes object that stores **non-secret** configuration as key/value pairs. Values can be short strings (`LOG_LEVEL=INFO`) or whole files (`nginx.conf`).

### Why do we need it?
**Build once, run anywhere.** The same container image runs in dev, UAT and prod. Only the configuration changes. Without ConfigMaps, people bake config into the image, so they need a new image for every environment — slow and risky.

### How it works internally
```
 kubectl apply ConfigMap --> kube-apiserver --> stored in etcd (/registry/configmaps/<ns>/<name>)
                                                    |
 Pod starts --> kubelet reads the ConfigMap via the API (watch/cache)
             --> as ENV VARS: copied once at container start (never updated later)
             --> as FILES (volume): written to the node, updated by kubelet when the ConfigMap changes
                 (delay: kubelet sync period + cache TTL, usually up to ~60-90 s)
```
Limit: a ConfigMap can hold at most **1 MiB**. For bigger files, use a volume or object storage.

### Real production scenario
Your Laravel app runs on OKE in UAT and PROD. `APP_ENV`, `CACHE_DRIVER` and the API base URL differ per environment. You keep one image `app:2.4.1` and two ConfigMaps (`app-config` in `uat` and in `prod` namespaces). Promoting a release means changing only the image tag.

### Hands-on lab 1 — three ways to create a ConfigMap

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"

**Way 1: from literals** (key=value on the command line). Good for quick tests.

In [ ]:
%%bash
kubectl create configmap app-settings \
  --from-literal=APP_ENV=uat \
  --from-literal=LOG_LEVEL=debug \
  --dry-run=client -o yaml | kubectl apply -f -
kubectl get configmap app-settings -o yaml | sed -n '/^data:/,/^kind:/p'

**Expected output**
```
data:
  APP_ENV: uat
  LOG_LEVEL: debug
kind: ConfigMap
```
**Way 2: from a file.** The file name becomes the key, the file content becomes the value.

In [ ]:
%%writefile labs/06/nginx.conf
server {
    listen 8080;
    location / {
        return 200 "hello from configmap\n";
    }
    location /healthz {
        return 200 "ok\n";
    }
}

In [ ]:
%%bash
kubectl create configmap nginx-conf --from-file=default.conf=labs/06/nginx.conf --dry-run=client -o yaml | kubectl apply -f -
kubectl describe configmap nginx-conf | head -20

**Way 3: declarative YAML** (best for Git / GitOps).

In [ ]:
%%writefile labs/06/configmap.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: shop-config
  namespace: lab-06
  labels:
    app: shop
data:                          # text values (UTF-8)
  APP_ENV: "uat"               # always quote values: "true", "8080" stay strings
  FEATURE_NEW_CHECKOUT: "false"
  API_BASE_URL: "http://payments.lab-06.svc.cluster.local:8080"
  app.yaml: |                  # a whole file as one value ("|" keeps new lines)
    cache:
      ttlSeconds: 300
    retries: 3
# binaryData: {}               # base64 values for binary files (rarely needed)

In [ ]:
%%bash
kubectl apply -f labs/06/configmap.yaml
kubectl get configmap

**Expected output**
```
NAME               DATA   AGE
app-settings       2      1m
kube-root-ca.crt   1      5m    <- created automatically in every namespace (cluster CA cert)
nginx-conf         1      1m
shop-config        4      2s
```

### Troubleshooting ConfigMaps
| Symptom | Cause | Fix |
|---|---|---|
| Pod `CreateContainerConfigError: configmap "x" not found` | ConfigMap missing or in another namespace | Create it in the Pod's namespace, or mark it `optional: true` |
| App reads `8080` as a number / YAML error on `true` | Unquoted value | Quote all values in `data:` |
| `Too long: must have at most 1048576 bytes` | Over 1 MiB | Store large files elsewhere |
| Changed the ConfigMap, app still uses old value | Env vars never update; app does not reload files | Restart the Deployment, or use hash-suffix rollouts (Topic 7) |

### Lab vs Production
| Lab | Production |
|---|---|
| `kubectl create --from-literal` | YAML in Git, applied by CI/CD or Argo CD |
| One namespace | One ConfigMap per environment (namespace or overlay) |
| Mutable ConfigMaps | Immutable or hash-suffixed ConfigMaps for safe rollouts and rollbacks |

### Check yourself
1. Should a database password go in a ConfigMap?
<details><summary>Answer</summary>No. Use a Secret (better: External Secrets from a vault). ConfigMaps are for non-secret data and are readable by anyone with ConfigMap read access.</details>

2. What becomes the key when you use `--from-file=nginx.conf`?
<details><summary>Answer</summary>The file name <code>nginx.conf</code> (or the name you give: <code>--from-file=default.conf=nginx.conf</code>).</details>

3. What is `kube-root-ca.crt`?
<details><summary>Answer</summary>A ConfigMap created automatically in every namespace with the cluster CA certificate, so Pods can verify the API server's TLS certificate.</details>

---
# Topic 2 — Secrets (base64 is NOT encryption)

### What is it?
A **Secret** stores sensitive data: passwords, API tokens, TLS certificates, registry credentials. It looks like a ConfigMap, but:
- values are **base64-encoded** in the API,
- secret volumes use **tmpfs** (RAM), not disk,
- RBAC is usually stricter on Secrets,
- they **can** be encrypted at rest in etcd (Topic 6) — but **not by default** on self-managed clusters.

### Common Secret types
| `type` | Use |
|---|---|
| `Opaque` | generic key/values (default) |
| `kubernetes.io/tls` | TLS cert + key (`tls.crt`, `tls.key`) for Ingress |
| `kubernetes.io/dockerconfigjson` | image pull credentials (e.g. OCI Registry / OCIR) |
| `kubernetes.io/service-account-token` | long-lived SA token (legacy; avoid) |
| `kubernetes.io/basic-auth`, `ssh-auth` | username/password, SSH key |

### Why base64 is not security
**Base64** is an **encoding**: a way to write bytes as text. Anyone can decode it. There is no key. It exists so binary data (certificates) can live in JSON/YAML.

### How it works internally
```
 Secret YAML (base64) --> kube-apiserver --> etcd
                                      |
                       no EncryptionConfiguration?  --> stored as PLAIN TEXT in etcd
                       with EncryptionConfiguration --> stored encrypted (aescbc/secretbox/KMS)
                                      |
 kubelet fetches only the Secrets used by Pods on ITS node (NodeAuthorizer)
   --> writes them to tmpfs --> mounts into the container, or sets env vars
```

### Real production scenario
A developer commits `secret.yaml` with `DB_PASSWORD: UHJvZFBhc3MxMjM=` to Azure DevOps Repos and says "it is encrypted". Anyone with repo access can run `base64 -d` and get the production password. The fix: rotate the password, purge it from Git history, and move secrets to OCI Vault + External Secrets.

### Hands-on lab 2.1 — decode a Secret in one command

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"
kubectl create secret generic db-secret \
  --from-literal=username=shop_app \
  --from-literal=password='Lab-Fake-Pass-42' \
  --dry-run=client -o yaml | kubectl apply -f -
echo "--- What the API stores (base64):"
kubectl get secret db-secret -o jsonpath='{.data.password}'; echo
echo "--- Anyone with 'get secret' permission can decode it:"
kubectl get secret db-secret -o jsonpath='{.data.password}' | base64 -d; echo

**Expected output**
```
secret/db-secret created
--- What the API stores (base64):
TGFiLUZha2UtUGFzcy00Mg==
--- Anyone with 'get secret' permission can decode it:
Lab-Fake-Pass-42
```
**Lesson:** protect Secrets with **RBAC** (who can `get`/`list` secrets) and **encryption at rest**. Note: `list secrets` permission also reveals all values.

### Hands-on lab 2.2 — write a Secret in YAML with `stringData`

`stringData` lets you write plain text. The API server encodes it into `data` for you. It is easier to read, but the file is still plain text — never commit it.

In [ ]:
%%writefile labs/06/secret.yaml
apiVersion: v1
kind: Secret
metadata:
  name: payment-api
  namespace: lab-06
type: Opaque                   # generic key/value secret
stringData:                    # plain text here; API server stores it base64-encoded in .data
  API_KEY: "lab-fake-key-0001"
  WEBHOOK_SECRET: "lab-fake-webhook-xyz"
# data:                        # alternative: values already base64-encoded
#   API_KEY: bGFiLWZha2Uta2V5LTAwMDE=
immutable: false               # true = cannot be changed (see Topic 7)

In [ ]:
%%bash
kubectl apply -f labs/06/secret.yaml
kubectl get secret payment-api -o jsonpath='{.data}'; echo
kubectl get secret payment-api -o yaml | grep -c stringData || echo "stringData is not stored: only .data exists"

**Expected output**
```
{"API_KEY":"bGFiLWZha2Uta2V5LTAwMDE=","WEBHOOK_SECRET":"bGFiLWZha2Utd2ViaG9vay14eXo="}
0
stringData is not stored: only .data exists
```

### Hands-on lab 2.3 — a TLS Secret and an image pull Secret
We create a self-signed certificate with `openssl` (macOS has it) and store it as a `kubernetes.io/tls` Secret.

In [ ]:
%%bash
cd labs/06
openssl req -x509 -nodes -newkey rsa:2048 -days 30 -subj "/CN=shop.lab.local" \
  -keyout tls.key -out tls.crt 2>/dev/null
kubectl create secret tls shop-tls --cert=tls.crt --key=tls.key --dry-run=client -o yaml | kubectl apply -f -
kubectl get secret shop-tls -o jsonpath='{.type}{"  keys: "}{.data}' | cut -c1-80; echo
# Image pull secret (fake credentials) - the format used for OCIR, Docker Hub, ACR...
kubectl create secret docker-registry ocir-pull \
  --docker-server=me-jeddah-1.ocir.io \
  --docker-username='tenancy-namespace/fake.user@example.com' \
  --docker-password='fake-auth-token' \
  --dry-run=client -o yaml | kubectl apply -f -
kubectl get secret ocir-pull -o jsonpath='{.type}'; echo
rm -f tls.key   # do not leave private keys lying around

**Expected output**
```
secret/shop-tls created
kubernetes.io/tls  keys: {"tls.crt":"LS0tLS1CRUdJTi...","tls.key":"LS0t...
secret/ocir-pull created
kubernetes.io/dockerconfigjson
```
A Pod uses the pull secret with `spec.imagePullSecrets: [{name: ocir-pull}]`, or you attach it to the ServiceAccount.

### Troubleshooting Secrets
| Symptom | Cause | Fix |
|---|---|---|
| `ImagePullBackOff` with `unauthorized` | pull secret missing, wrong namespace, or expired auth token | Create it in the Pod's namespace; refresh OCIR auth token |
| App gets `\n` at the end of the password | Created with `echo "pass" \| base64` (adds newline) | Use `echo -n` or `--from-literal` |
| `illegal base64 data` on apply | Plain text put in `data:` | Use `stringData:` or encode correctly |
| Ingress TLS not working | Secret not type `kubernetes.io/tls` or keys not `tls.crt`/`tls.key` | Recreate with `kubectl create secret tls` |

### Lab vs Production
| Lab | Production |
|---|---|
| Secrets from literals | Secrets synced from OCI Vault / HashiCorp Vault via External Secrets (Topic 5) |
| Self-signed TLS | cert-manager + Let's Encrypt / internal CA, auto-renewal |
| Anyone in the namespace can read | RBAC: only the app's ServiceAccount and a few admins; audit logging on secret reads |
| etcd plain text (kind default) | Encryption at rest with KMS (OKE: OCI Vault key) |

### Check yourself
1. Is base64 encryption?
<details><summary>Answer</summary>No. It is an encoding with no key. Anyone can decode it with <code>base64 -d</code>.</details>

2. What is the difference between `data` and `stringData`?
<details><summary>Answer</summary><code>data</code> needs base64-encoded values. <code>stringData</code> takes plain text; the API server encodes it into <code>data</code>. Only <code>data</code> is stored.</details>

3. Which permission is as dangerous as `get secrets`?
<details><summary>Answer</summary><code>list secrets</code> (it returns all values), and also <code>create pods</code> in that namespace (a Pod can mount any Secret there).</details>

---
# Topic 3 — Environment variables: env, envFrom, valueFrom, Downward API

### What is it?
Ways to pass configuration to a container as **environment variables**:
| Method | What it does |
|---|---|
| `env: [{name, value}]` | a fixed value in the Pod spec |
| `env[].valueFrom.configMapKeyRef` / `secretKeyRef` | one key from a ConfigMap / Secret |
| `envFrom: [configMapRef / secretRef]` | **all** keys become variables (optional `prefix`) |
| `env[].valueFrom.fieldRef` | Pod information: name, namespace, node name, Pod IP, labels... (**Downward API**) |
| `env[].valueFrom.resourceFieldRef` | the container's own CPU/memory requests and limits |

### Why do we need it?
Most apps (12-factor apps) read config from environment variables. The **Downward API** lets the app know about itself without calling the Kubernetes API (no extra permissions). Example: a Java app sets its heap size from its memory limit.

### How it works internally
```
 kubelet, at CONTAINER START:
   resolves every env entry (reads ConfigMaps/Secrets, Pod fields)
   --> passes the final list to the container runtime --> process environment
 After start: values are FROZEN. Changing the ConfigMap does nothing until the container restarts.
```
Order rule: if `envFrom` and `env` define the same name, **`env` wins**.

### Real production scenario
Your Java service on OKE gets `OOMKilled` because the JVM heap is bigger than the container limit. You pass the memory limit with `resourceFieldRef` and start Java with `-XX:MaxRAMPercentage=75`. You also log `POD_NAME` and `NODE_NAME` in every log line, so you can find which node had the problem in Grafana/OCI Logging.

### Hands-on lab 3 — all env methods in one Pod

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"

In [ ]:
%%writefile labs/06/env-demo.yaml
apiVersion: v1
kind: Pod
metadata:
  name: env-demo
  namespace: lab-06
  labels:
    app: env-demo
    version: "2.4.1"
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "env | sort | grep -E '^(APP_|CFG_|DB_|POD_|NODE_|MEM_|CPU_|GREETING)'; sleep 3600"]
      resources:
        requests: {cpu: 100m, memory: 64Mi}
        limits:   {cpu: 200m, memory: 128Mi}
      envFrom:
        - configMapRef:
            name: shop-config        # ALL keys of the ConfigMap become variables
          prefix: CFG_               # ... with this prefix: CFG_APP_ENV, CFG_API_BASE_URL
        # keys that are not valid variable names (like "app.yaml") are SKIPPED
      env:
        - name: GREETING             # plain value
          value: "hello"
        - name: APP_ENV              # one key from a ConfigMap
          valueFrom:
            configMapKeyRef:
              name: app-settings
              key: APP_ENV
        - name: DB_PASSWORD          # one key from a Secret
          valueFrom:
            secretKeyRef:
              name: db-secret
              key: password
        - name: DB_OPTIONAL          # missing key is OK because optional: true
          valueFrom:
            secretKeyRef:
              name: db-secret
              key: does-not-exist
              optional: true
        - name: POD_NAME             # Downward API: Pod fields
          valueFrom:
            fieldRef: {fieldPath: metadata.name}
        - name: POD_NAMESPACE
          valueFrom:
            fieldRef: {fieldPath: metadata.namespace}
        - name: POD_IP
          valueFrom:
            fieldRef: {fieldPath: status.podIP}
        - name: NODE_NAME
          valueFrom:
            fieldRef: {fieldPath: spec.nodeName}
        - name: POD_VERSION_LABEL
          valueFrom:
            fieldRef: {fieldPath: "metadata.labels['version']"}
        - name: MEM_LIMIT_MB         # Downward API: own resource limits
          valueFrom:
            resourceFieldRef:
              containerName: app
              resource: limits.memory
              divisor: 1Mi           # report in MiB instead of bytes
        - name: CPU_REQUEST_MILLI
          valueFrom:
            resourceFieldRef:
              resource: requests.cpu
              divisor: 1m            # report in millicores

In [ ]:
%%bash
kubectl apply -f labs/06/env-demo.yaml
kubectl wait --for=condition=Ready pod/env-demo --timeout=90s
kubectl logs env-demo

**Expected output**
```
APP_ENV=uat
CFG_API_BASE_URL=http://payments.lab-06.svc.cluster.local:8080
CFG_APP_ENV=uat
CFG_FEATURE_NEW_CHECKOUT=false
CPU_REQUEST_MILLI=100
DB_PASSWORD=Lab-Fake-Pass-42
GREETING=hello
MEM_LIMIT_MB=128
NODE_NAME=learn-worker
POD_IP=10.244.1.7
POD_NAMESPACE=lab-06
POD_NAME=env-demo
POD_VERSION_LABEL=2.4.1
```
Notes:
- `app.yaml` was skipped (a dot is not allowed in a variable name). `kubectl describe` does not warn you; check events with `kubectl get events`.
- `DB_OPTIONAL` is not set, and the Pod still started, because of `optional: true`.
- `DB_PASSWORD` is visible to anything that can read `/proc/1/environ` or `kubectl exec env`. Files are a bit safer than env vars for secrets.

Now prove that env vars **do not update**:

In [ ]:
%%bash
kubectl patch configmap app-settings -p '{"data":{"APP_ENV":"prod"}}'
sleep 5
kubectl exec env-demo -- sh -c 'echo "APP_ENV inside the running container: $APP_ENV"'

**Expected output**
```
configmap/app-settings patched
APP_ENV inside the running container: uat
```
Still `uat`. Only a **new container** (restart / rollout) sees `prod`.

### Troubleshooting env vars
| Symptom | Cause | Fix |
|---|---|---|
| `CreateContainerConfigError: couldn't find key X in Secret` | Key missing | Fix the key name or add `optional: true` |
| Variable missing with `envFrom` | Key is not a valid env name (has `.` or `-` at start) | Rename the key, or mount as a file |
| Wrong value | Same name in `env` and `envFrom` → `env` wins | Remove the duplicate |
| Changed ConfigMap, no effect | Env vars are frozen at start | `kubectl rollout restart deployment/<name>` |

### Lab vs Production
| Lab | Production |
|---|---|
| Secret as env var | Prefer secret **files** (not in `env` output, not leaked by crash dumps that print the environment) |
| Manual restart after change | Automated restart via hash-suffix ConfigMaps or a reloader controller |
| Downward API for learning | Use it for logging context (`POD_NAME`, `NODE_NAME`) and JVM/Go memory tuning |

### Check yourself
1. `envFrom` gives `LOG_LEVEL=info`, `env` gives `LOG_LEVEL=debug`. Which value wins?
<details><summary>Answer</summary><code>debug</code>. Values in <code>env</code> override <code>envFrom</code>.</details>

2. How can an app learn its own memory limit without calling the Kubernetes API?
<details><summary>Answer</summary>Downward API: <code>resourceFieldRef</code> with <code>resource: limits.memory</code>.</details>

3. Why are env vars a weaker place for secrets than files?
<details><summary>Answer</summary>They are visible in <code>/proc/&lt;pid&gt;/environ</code>, inherited by child processes, and often printed in debug logs or crash reports. They also never rotate without a restart.</details>

---
# Topic 4 — Mounting configuration as files (and the subPath gotcha)

### What is it?
You can mount a ConfigMap or Secret as a **volume**. Each key becomes a **file**. This is how you ship `nginx.conf`, `application.yaml` or certificates.

### Why do we need it?
- Many programs read config **files**, not env vars.
- Mounted files **update automatically** when the ConfigMap changes (env vars do not).
- Secrets as files stay on tmpfs and can be rotated without restarting (if the app re-reads them).

### How it works internally — why updates work, and why `subPath` breaks them
```
 /etc/nginx/conf.d/                      (mountPath, the volume root)
   default.conf -> ..data/default.conf   (symlink)
   ..data       -> ..2026_10_27_10_00_00.123   (symlink to the CURRENT version directory)
   ..2026_10_27_10_00_00.123/default.conf

 ConfigMap changes --> kubelet writes a NEW timestamp directory
                   --> atomically swaps the "..data" symlink   (no half-written files)

 subPath: mounts ONE FILE directly (bind mount of the real file at container start)
          --> there is no symlink to swap --> the file NEVER updates
```
Also: mounting a volume on a directory **hides** everything that was in that directory in the image. `subPath` is often used to avoid that — and then auto-update is lost.

### Real production scenario
You change the NGINX config ConfigMap on OKE to add a new location. Half of your Pods pick it up after a minute (they mount the directory). The other half never do (they used `subPath` to put `nginx.conf` into `/etc/nginx/`). Users get different behaviour depending on the Pod. The fix: mount into `/etc/nginx/conf.d/` (directory) and trigger a rollout on config change.

### Hands-on lab 4 — directory mount vs subPath mount, side by side

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"

In [ ]:
%%writefile labs/06/mounts.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: live-config
  namespace: lab-06
data:
  message.txt: "version 1"
---
apiVersion: v1
kind: Pod
metadata:
  name: mount-demo
  namespace: lab-06
spec:
  volumes:
    - name: cfg
      configMap:
        name: live-config
        defaultMode: 0444            # read-only files for everyone
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      volumeMounts:
        - name: cfg                   # 1) whole directory: UPDATES automatically
          mountPath: /config-dir
          readOnly: true
        - name: cfg                   # 2) single file with subPath: NEVER updates
          mountPath: /etc/app/message.txt
          subPath: message.txt        # key name inside the ConfigMap
          readOnly: true

In [ ]:
%%bash
kubectl apply -f labs/06/mounts.yaml
kubectl wait --for=condition=Ready pod/mount-demo --timeout=90s
echo "--- directory listing (see the ..data symlinks):"
kubectl exec mount-demo -- ls -la /config-dir
echo "--- before change:"
kubectl exec mount-demo -- sh -c 'echo "dir mount : $(cat /config-dir/message.txt)"; echo "subPath   : $(cat /etc/app/message.txt)"'

**Expected output**
```
--- directory listing (see the ..data symlinks):
drwxrwxrwx ... .
drwxr-xr-x ... ..2026_10_27_10_15_00.1234567
lrwxrwxrwx ... ..data -> ..2026_10_27_10_15_00.1234567
lrwxrwxrwx ... message.txt -> ..data/message.txt
--- before change:
dir mount : version 1
subPath   : version 1
```
Now change the ConfigMap and wait. kubelet needs up to about a minute.

In [ ]:
%%bash
kubectl patch configmap live-config -p '{"data":{"message.txt":"version 2"}}'
for i in $(seq 1 24); do
  v=$(kubectl exec mount-demo -- cat /config-dir/message.txt)
  [ "$v" = "version 2" ] && { echo "dir mount updated after ~$((i*5))s"; break; }
  sleep 5
done
kubectl exec mount-demo -- sh -c 'echo "dir mount : $(cat /config-dir/message.txt)"; echo "subPath   : $(cat /etc/app/message.txt)"'

**Expected output**
```
configmap/live-config patched
dir mount updated after ~45s
dir mount : version 2
subPath   : version 1      <- the gotcha
```
**Important:** the file changed, but the **application** must also re-read it. NGINX needs `nginx -s reload`; many apps read config only at start. That is why the usual production pattern is a **rollout** on config change (Topic 7).

### Troubleshooting mounted config
| Symptom | Cause | Fix |
|---|---|---|
| File never updates | `subPath` mount, or env var | Mount the directory; or roll out on change |
| File updated, app behaves the same | App reads config only at start | Reload signal, sidecar reloader, or rollout |
| Image files in `/etc/nginx/` disappeared | Volume mounted over the whole directory | Mount to a sub-directory (`conf.d`) or use `subPath` knowingly |
| `permission denied` reading a secret file | `defaultMode` too strict for the app user | Set `defaultMode`/`fsGroup` correctly (e.g. 0440 + fsGroup) |
| Only some keys appear | `items:` limits which keys are mounted | Add the keys to `items` or remove `items` |

### Lab vs Production
| Lab | Production |
|---|---|
| Watch files update by hand | Automatic rollout when config changes (hash suffix / Reloader / Argo CD) |
| `subPath` demo | Avoid `subPath` for config that changes; document it if used |
| `0444` mode | Secrets `0400`/`0440` with `fsGroup`, owned by the app user |

### Check yourself
1. Why does a `subPath` mount not update?
<details><summary>Answer</summary>It bind-mounts the actual file at container start. Updates work by swapping the <code>..data</code> symlink in the volume directory, and a single-file bind mount does not follow that swap.</details>

2. The ConfigMap file updated inside the Pod, but NGINX still serves old behaviour. Why?
<details><summary>Answer</summary>NGINX reads its config at start. It needs a reload (<code>nginx -s reload</code>) or a Pod restart.</details>

3. What happens to files in the image at `/etc/nginx` if you mount a ConfigMap at `/etc/nginx`?
<details><summary>Answer</summary>They are hidden by the mount. Only the ConfigMap keys are visible there.</details>

---
# Topic 5 — External Secrets Operator (ESO)

### What is it?
The **External Secrets Operator** is a Kubernetes controller. It reads secrets from an **external secret manager** (OCI Vault, HashiCorp Vault, AWS Secrets Manager, Azure Key Vault...) and creates normal Kubernetes Secrets from them. It keeps them **in sync**.

Main objects:
| Object | Role |
|---|---|
| `SecretStore` (namespaced) / `ClusterSecretStore` (cluster-wide) | **how** to connect to the provider (which vault, which auth) |
| `ExternalSecret` | **what** to fetch and which Kubernetes Secret to create |

### Why do we need it?
- The **source of truth** is the vault: audited, access-controlled, rotated, backed up.
- Git contains only `ExternalSecret` objects (names of secrets, **no values**). Safe for GitOps.
- Rotation: change the value in the vault → ESO updates the Kubernetes Secret within `refreshInterval`.

### How it works internally
```
  OCI Vault  (secret "prod-db-password", version 3)
      ^
      |  ESO controller calls the provider API (auth: OKE Workload Identity / Instance Principal)
      |
  ExternalSecret "db-credentials"  --(every refreshInterval)-->  reconcile loop
      |
      v
  Kubernetes Secret "db-credentials"  (owned by the ExternalSecret)
      |
      v
  Pods mount / use it as usual  (pods do not talk to the vault at all)
```

### Real production scenario
Your security team requires all production passwords to live in **OCI Vault** with audit logs. Today, passwords are created by hand with `kubectl create secret`. You install ESO on OKE, create a `ClusterSecretStore` for OCI Vault using **Workload Identity**, and convert each Secret into an `ExternalSecret` committed to Git. Password rotation becomes: update the vault secret version; ESO syncs it within 1 hour; trigger a rollout.

### Hands-on lab 5 — install ESO and use the **Fake** provider (runs fully local)

ESO has a `fake` provider: the "vault" data is written inside the SecretStore. It is only for learning and tests, but every other part (sync, refresh, templating) is real.

Install ESO with Helm. This takes about 1 minute.

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"
helm repo add external-secrets https://charts.external-secrets.io >/dev/null 2>&1 || true
helm repo update external-secrets >/dev/null
helm upgrade --install external-secrets external-secrets/external-secrets \
  -n external-secrets --create-namespace --wait --timeout 5m
kubectl -n external-secrets get pods
kubectl api-resources | grep -E 'externalsecrets|secretstores' | head -4

**Expected output**
```
NAME                                                READY   STATUS    RESTARTS   AGE
external-secrets-...                                1/1     Running   0          60s
external-secrets-cert-controller-...                1/1     Running   0          60s
external-secrets-webhook-...                        1/1     Running   0          60s
clustersecretstores    css   external-secrets.io/v1   false   ClusterSecretStore
externalsecrets        es    external-secrets.io/v1   true    ExternalSecret
secretstores           ss    external-secrets.io/v1   true    SecretStore
```
> If your chart version shows `external-secrets.io/v1beta1` only, change `apiVersion` in the YAML below to `external-secrets.io/v1beta1`.

In [ ]:
%%writefile labs/06/eso-fake.yaml
apiVersion: external-secrets.io/v1
kind: SecretStore
metadata:
  name: fake-vault
  namespace: lab-06
spec:
  provider:
    fake:                         # FAKE provider: the "vault" content lives here (lab only!)
      data:
        - key: /prod/shop/db      # like a secret path in a real vault
          value: '{"username":"shop_app","password":"Vault-Fake-Pass-v1"}'
          version: v1
---
apiVersion: external-secrets.io/v1
kind: ExternalSecret
metadata:
  name: shop-db
  namespace: lab-06
spec:
  refreshInterval: 30s            # how often ESO re-reads the provider (prod: 1h is common)
  secretStoreRef:
    name: fake-vault
    kind: SecretStore             # or ClusterSecretStore
  target:
    name: shop-db-credentials     # the Kubernetes Secret ESO will create
    creationPolicy: Owner         # ESO owns it; deleting the ExternalSecret deletes the Secret
    template:                     # optional: shape the final Secret
      type: Opaque
      data:
        DB_USER: "{{ .username }}"
        DB_PASSWORD: "{{ .password }}"
        DB_URL: "mysql://{{ .username }}@mysql.lab-06.svc:3306/shop"
  dataFrom:
    - extract:                    # read a JSON secret and expose each field (username, password)
        key: /prod/shop/db
        version: v1

In [ ]:
%%bash
kubectl apply -f labs/06/eso-fake.yaml
sleep 10
kubectl get secretstore fake-vault
kubectl get externalsecret shop-db
echo "--- Kubernetes Secret created by ESO:"
kubectl get secret shop-db-credentials -o jsonpath='{.data.DB_PASSWORD}' | base64 -d; echo
kubectl get secret shop-db-credentials -o jsonpath='{.data.DB_URL}' | base64 -d; echo

**Expected output**
```
NAME         AGE   STATUS   CAPABILITIES   READY
fake-vault   10s   Valid    ReadWrite      True
NAME      STORETYPE     STORE        REFRESH INTERVAL   STATUS         READY
shop-db   SecretStore   fake-vault   30s                SecretSynced   True
--- Kubernetes Secret created by ESO:
Vault-Fake-Pass-v1
mysql://shop_app@mysql.lab-06.svc:3306/shop
```
Now **rotate** the password "in the vault" (we edit the fake store: version v2) and watch ESO sync it.

In [ ]:
%%bash
kubectl patch secretstore fake-vault --type=json -p='[
 {"op":"replace","path":"/spec/provider/fake/data/0/value","value":"{\"username\":\"shop_app\",\"password\":\"Vault-Fake-Pass-v2\"}"},
 {"op":"replace","path":"/spec/provider/fake/data/0/version","value":"v2"}]'
kubectl patch externalsecret shop-db --type=json -p='[{"op":"replace","path":"/spec/dataFrom/0/extract/version","value":"v2"}]'
for i in $(seq 1 12); do
  p=$(kubectl get secret shop-db-credentials -o jsonpath='{.data.DB_PASSWORD}' | base64 -d)
  [ "$p" = "Vault-Fake-Pass-v2" ] && { echo "Rotated after ~$((i*5))s: $p"; break; }
  sleep 5
done

**Expected output**
```
Rotated after ~5s: Vault-Fake-Pass-v2
```
Remember: Pods that read the Secret as **env vars** still have the old value until they restart.

### Production example — OCI Vault on OKE (do not apply here)
```yaml
apiVersion: external-secrets.io/v1
kind: ClusterSecretStore
metadata:
  name: oci-vault
spec:
  provider:
    oracle:
      vault: ocid1.vault.oc1.me-jeddah-1.<unique-id>      # your Vault OCID
      region: me-jeddah-1
      principalType: Workload       # OKE Workload Identity (no API keys in the cluster)
      serviceAccountRef:
        name: external-secrets
        namespace: external-secrets
---
apiVersion: external-secrets.io/v1
kind: ExternalSecret
metadata:
  name: shop-db
  namespace: shop-prod
spec:
  refreshInterval: 1h
  secretStoreRef: {name: oci-vault, kind: ClusterSecretStore}
  target: {name: shop-db-credentials, creationPolicy: Owner}
  data:
    - secretKey: DB_PASSWORD
      remoteRef:
        key: shop-prod-db-password   # secret name in OCI Vault
```
The IAM side: an OCI policy that allows the workload (cluster + namespace + service account) to read secret bundles in the vault's compartment, for example:
`Allow any-user to read secret-bundles in compartment <app-compartment> where all {request.principal.type='workload', request.principal.namespace='external-secrets', request.principal.service_account='external-secrets', request.principal.cluster_id='<cluster OCID>'}`
Check the exact field names in the current ESO Oracle provider docs before production use.

### Troubleshooting ESO
| Symptom | Cause | Fix |
|---|---|---|
| SecretStore `InvalidProviderConfig` / not Ready | Wrong vault OCID/region, auth failed | `kubectl describe secretstore`; check IAM policy and Workload Identity |
| ExternalSecret `SecretSyncedError: ... not found` | Wrong remote key name or version | Compare with the vault; check the case |
| Secret not updated after rotation | `refreshInterval` long, or pinned `version` | Wait / use latest version / annotate `force-sync=$(date +%s)` |
| Secret deleted unexpectedly | `creationPolicy: Owner` and ExternalSecret deleted | Use `deletionPolicy: Retain` in `target` if needed |
| Webhook errors on apply | ESO webhook not ready | `kubectl -n external-secrets get pods`, wait for Ready |

### Lab vs Production
| Lab | Production |
|---|---|
| Fake provider with values in YAML | OCI Vault / HashiCorp Vault; values never in Git |
| `refreshInterval: 30s` | 1h typical (API cost, rate limits); force-sync on demand |
| Namespaced SecretStore | ClusterSecretStore managed by the platform team, RBAC to limit which namespaces may use it |
| No monitoring | Alert on ESO metrics `externalsecret_status_condition` (SecretSyncedError) |

### Check yourself
1. Does the application Pod talk to the vault when ESO is used?
<details><summary>Answer</summary>No. ESO talks to the vault and writes a normal Kubernetes Secret. The Pod reads that Secret as usual.</details>

2. What is safe to commit to Git: the `ExternalSecret` or the resulting `Secret`?
<details><summary>Answer</summary>The <code>ExternalSecret</code>. It contains only references (key names), no secret values.</details>

3. SecretStore vs ClusterSecretStore?
<details><summary>Answer</summary>SecretStore works only in its own namespace. ClusterSecretStore can be referenced from any namespace (restrict with conditions/RBAC).</details>

---
# Topic 6 — Secret encryption at rest in etcd

### What is it?
**etcd** is the database of the cluster. Every Secret is stored there. **Encryption at rest** means the API server encrypts Secrets **before** writing them to etcd. Anyone who copies the etcd data file or an etcd backup sees only ciphertext.

### Why do we need it?
Without it, an etcd **backup file** or a stolen control-plane disk contains every password of the cluster in plain text. Compliance standards (ISO 27001, NCA ECC in Saudi Arabia, PCI DSS) expect encryption at rest.

### How it works internally
```
 kube-apiserver --encryption-provider-config=/etc/kubernetes/enc/config.yaml

 Write: Secret --> first provider in the list encrypts --> etcd stores "k8s:enc:<provider>:v1:<keyname>:<ciphertext>"
 Read:  etcd --> API server tries providers in order --> plaintext to the client

 Providers:
   identity        = no encryption (default)
   aescbc / aesgcm / secretbox = local key in the config file on the control plane
   kms (v2)        = "envelope encryption": a DEK (data key) per write, encrypted by a KEK in an
                     external KMS (OCI Vault, AWS KMS, Azure Key Vault). The KEK never leaves the KMS.
```
On **OKE**, you do not touch the API server. You select an **OCI Vault master encryption key** when you create (or update) the cluster. OKE then encrypts Kubernetes Secrets in etcd with envelope encryption using that key. You also need an IAM policy that lets the cluster use the key.

### Real production scenario
An auditor asks: "If someone gets your etcd backup, can they read your database passwords?" On kind (no encryption): yes. On OKE with an OCI Vault key: no — they also need permission to use the key in OCI Vault, which is audited.

### Hands-on lab 6 — see the plain text in etcd (kind default)

We read the raw etcd value of our Secret with `etcdctl`, which runs inside the etcd Pod on the control plane. This shows what an attacker with an etcd backup would see.

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"
kubectl -n kube-system exec etcd-learn-control-plane -- etcdctl \
  --endpoints=https://127.0.0.1:2379 \
  --cacert=/etc/kubernetes/pki/etcd/ca.crt \
  --cert=/etc/kubernetes/pki/etcd/server.crt \
  --key=/etc/kubernetes/pki/etcd/server.key \
  get /registry/secrets/lab-06/db-secret | LC_ALL=C tr -c '[:print:]\n' '.' | head -c 400; echo

**Expected output** (binary parts shown as dots):
```
/registry/secrets/lab-06/db-secret
k8s.....v1..Secret.......db-secret..lab-06...........password..Lab-Fake-Pass-42..username..shop_app..Opaque...
```
The password `Lab-Fake-Pass-42` is **readable**. There is no `k8s:enc:` prefix, so nothing is encrypted.

### What enabling encryption looks like (self-managed / kubeadm)
This is the config file you would give the API server. **We do not apply it to `learn`** — it needs a control-plane restart. (Optional extra lab: create a second kind cluster with this file mounted; see the assignment hints.)

```yaml
apiVersion: apiserver.config.k8s.io/v1
kind: EncryptionConfiguration
resources:
  - resources: ["secrets"]          # what to encrypt (can add configmaps)
    providers:
      - kms:                        # 1st provider = used for NEW writes
          apiVersion: v2
          name: oci-kms-plugin
          endpoint: unix:///var/run/kms-plugin/socket.sock   # local KMS plugin talks to the cloud KMS
          timeout: 3s
      - aescbc:                     # fallback / old key: still able to READ old data
          keys:
            - name: key1
              secret: <32-byte base64 key>
      - identity: {}                # read data that was never encrypted
```
After enabling, **existing** Secrets are still plain text until rewritten:
```
kubectl get secrets -A -o json | kubectl replace -f -
```

### Troubleshooting encryption at rest
| Symptom | Cause | Fix |
|---|---|---|
| Old secrets still plain in etcd | Encryption applies to new writes only | Rewrite all secrets (`kubectl replace`) |
| API server fails to start after config change | Bad file path / YAML / key length | Check API server logs; key must be 32 bytes for aescbc |
| Secrets unreadable after key removal | Removed a key still used by stored data | Never remove an old key before re-encrypting with the new one |
| OKE cluster cannot read secrets after key change | IAM policy for the key missing, key disabled/deleted | Restore policy; never delete a Vault key in use (use scheduled deletion carefully) |

### Lab vs Production
| Lab (kind) | Production |
|---|---|
| No encryption (identity) | KMS v2 envelope encryption; OKE: OCI Vault master key |
| etcd reachable with `kubectl exec` | etcd not reachable by users (managed control plane on OKE) |
| No key rotation | Rotate keys regularly; keep old key until all data re-encrypted |

### Check yourself
1. Does encryption at rest stop a user with `get secrets` permission from reading passwords?
<details><summary>Answer</summary>No. The API server decrypts for authorized clients. Encryption at rest protects etcd files and backups. RBAC protects API access.</details>

2. After you enable encryption, are old Secrets encrypted automatically?
<details><summary>Answer</summary>No. Only on the next write. Rewrite them all with <code>kubectl get secrets -A -o json | kubectl replace -f -</code>.</details>

3. What is envelope encryption?
<details><summary>Answer</summary>Data is encrypted with a data key (DEK); the DEK is encrypted with a key-encryption key (KEK) held in a KMS. The KEK never leaves the KMS.</details>

---
# Topic 7 — Configuration management patterns

### What is it?
Rules and tools that make config changes **safe, visible and reversible**:
1. **Immutable ConfigMaps/Secrets** (`immutable: true`): cannot be changed, only replaced.
2. **Hash-suffix names** (kustomize `configMapGenerator`): every content change creates a **new name** like `shop-config-7h9k2m4b5c`. The Deployment refers to the new name, so it **rolls out** automatically.
3. **Overlays per environment** (kustomize base + overlays for uat/prod).

### Why do we need it?
With a normal mutable ConfigMap:
- A bad change hits **all running Pods at once** (files) or **none until restart** (env vars). Both are bad.
- `kubectl rollout undo` does **not** bring the old config back — the ConfigMap is not versioned with the Deployment.
With hash-suffix names, config is part of the Deployment's Pod template: a change = a normal rolling update; rollback = old ReplicaSet with the old ConfigMap name.

Bonus: immutable ConfigMaps reduce load on the API server (kubelets stop watching them).

### How it works internally
```
 kustomization.yaml: configMapGenerator: app-config (from files/literals)
        |
 kubectl apply -k   --> kustomize computes hash of content --> name: app-config-<hash>
                    --> rewrites every reference in the Deployment to app-config-<hash>
        |
 content changes --> new hash --> new name --> Pod template changed --> rolling update
 old ConfigMap stays (until pruned) --> rollback still works
```

### Real production scenario
Someone sets `FEATURE_NEW_CHECKOUT=true` in the prod ConfigMap of an OKE app at 15:00. Payments fail. With env vars, nothing happened until a node drain at 22:00 restarted Pods — and nobody linked the outage to the 15:00 change. With hash-suffix ConfigMaps, the change would have triggered a visible rollout at 15:00, and `kubectl rollout undo` would fix it in seconds.

### Hands-on lab 7.1 — immutable ConfigMap

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }; echo "OK: $ctx"
cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: ConfigMap
metadata: {name: release-2-4-1, namespace: lab-06}
data: {FEATURE_NEW_CHECKOUT: "false"}
immutable: true          # cannot be edited after creation
EOF
kubectl patch configmap release-2-4-1 -p '{"data":{"FEATURE_NEW_CHECKOUT":"true"}}' 2>&1 | tail -1

**Expected output**
```
configmap/release-2-4-1 created
The ConfigMap "release-2-4-1" is invalid: data: Forbidden: field is immutable when `immutable` is set
```
To change it you create `release-2-4-2` and point the Deployment to it.

### Hands-on lab 7.2 — kustomize configMapGenerator with automatic rollout
`kubectl` has kustomize built in (`kubectl apply -k`). We build a small app folder.

In [ ]:
%%bash
mkdir -p labs/06/kustomize-app
cat > labs/06/kustomize-app/app.properties <<'EOF'
checkout.newFlow=false
cache.ttlSeconds=300
EOF

In [ ]:
%%writefile labs/06/kustomize-app/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: shop-web
spec:
  replicas: 2
  selector:
    matchLabels: {app: shop-web}
  template:
    metadata:
      labels: {app: shop-web}
    spec:
      containers:
        - name: web
          image: busybox:1.36
          command: ["sh", "-c", "cat /config/app.properties; sleep 3600"]
          volumeMounts:
            - name: cfg
              mountPath: /config
      volumes:
        - name: cfg
          configMap:
            name: shop-web-config      # kustomize rewrites this to shop-web-config-<hash>

In [ ]:
%%writefile labs/06/kustomize-app/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
namespace: lab-06                      # put everything in this namespace
resources:
  - deployment.yaml
configMapGenerator:
  - name: shop-web-config              # base name; final name gets a content hash suffix
    files:
      - app.properties
    options:
      immutable: true                  # each generated version is immutable
      labels:
        app: shop-web

In [ ]:
%%bash
echo "--- What kustomize generates (look at the name):"
kubectl kustomize labs/06/kustomize-app | grep -E 'name: shop-web-config'
kubectl apply -k labs/06/kustomize-app
kubectl rollout status deployment/shop-web --timeout=120s
kubectl get configmap -l app=shop-web
kubectl rollout history deployment/shop-web

**Expected output**
```
--- What kustomize generates (look at the name):
  name: shop-web-config-8m2t4h6k9b
          name: shop-web-config-8m2t4h6k9b
configmap/shop-web-config-8m2t4h6k9b created
deployment.apps/shop-web created
deployment "shop-web" successfully rolled out
NAME                         DATA   AGE
shop-web-config-8m2t4h6k9b   1      5s
REVISION  CHANGE-CAUSE
1         <none>
```
Now change the config file and apply again. Watch: a **new ConfigMap** and a **new rollout**.

In [ ]:
%%bash
sed -i.bak 's/checkout.newFlow=false/checkout.newFlow=true/' labs/06/kustomize-app/app.properties && rm labs/06/kustomize-app/app.properties.bak
kubectl apply -k labs/06/kustomize-app
kubectl rollout status deployment/shop-web --timeout=120s
kubectl get configmap -l app=shop-web
kubectl rollout history deployment/shop-web
POD=$(kubectl get pod -l app=shop-web -o jsonpath='{.items[0].metadata.name}')
kubectl logs "$POD" | head -1

**Expected output**
```
configmap/shop-web-config-5d7f2k8c4g created
deployment.apps/shop-web configured
deployment "shop-web" successfully rolled out
NAME                         DATA   AGE
shop-web-config-5d7f2k8c4g   1      5s
shop-web-config-8m2t4h6k9b   1      2m
REVISION  CHANGE-CAUSE
1         <none>
2         <none>
checkout.newFlow=true
```
Now the new flow "breaks payments". Roll back — the old ConfigMap comes back with the old ReplicaSet:

In [ ]:
%%bash
kubectl rollout undo deployment/shop-web
kubectl rollout status deployment/shop-web --timeout=120s
POD=$(kubectl get pod -l app=shop-web --sort-by=.metadata.creationTimestamp -o jsonpath='{.items[-1].metadata.name}')
kubectl logs "$POD" | head -1

**Expected output**
```
deployment.apps/shop-web rolled back
checkout.newFlow=false
```
Note: old generated ConfigMaps pile up. Clean them with `kubectl apply -k ... --prune` (careful) or let Argo CD prune old ones after a retention period.

### Troubleshooting config patterns
| Symptom | Cause | Fix |
|---|---|---|
| `field is immutable` on apply | Editing an immutable ConfigMap | Create a new name (generator does this) |
| Rollback restored image but not config | Plain mutable ConfigMap | Use hash-suffix names |
| Rollback fails: ConfigMap not found | Old generated ConfigMap was pruned | Keep old versions for N days |
| Hash changes on every apply | Generator input changes (e.g. a timestamp in a file) | Keep generated input deterministic |

### Lab vs Production
| Lab | Production |
|---|---|
| `kubectl apply -k` by hand | Argo CD / Flux applying kustomize or Helm from Git |
| One environment | `base/` + `overlays/uat`, `overlays/prod` with different generators |
| No cleanup | Retention policy for old generated ConfigMaps |

### Check yourself
1. Why does a hash-suffix ConfigMap trigger a rollout?
<details><summary>Answer</summary>The name changes, so the Deployment's Pod template changes (the volume/env reference), and any Pod template change creates a new ReplicaSet.</details>

2. Name two benefits of `immutable: true`.
<details><summary>Answer</summary>Protects against accidental changes, and reduces API server load because kubelets stop watching immutable objects.</details>

3. Why does `kubectl rollout undo` not restore a normal ConfigMap?
<details><summary>Answer</summary>ConfigMaps are separate objects, not versioned with the Deployment. Undo only restores the Pod template, which still points to the same (already changed) ConfigMap name.</details>

---
# Topic 8 — External config providers: Secrets Store CSI Driver

### What is it?
The **Secrets Store CSI Driver** mounts secrets from an external vault **directly into the Pod as files**, through a CSI volume. A **provider plugin** (OCI, Azure, AWS, GCP, Vault) talks to the vault.

Main object: `SecretProviderClass` — which vault, which secrets, how to authenticate.

### ESO vs Secrets Store CSI — when to use which
| | External Secrets Operator | Secrets Store CSI Driver |
|---|---|---|
| Result | Kubernetes **Secret** object | **files** in the Pod (optional sync to a Secret) |
| Secret stored in etcd? | Yes | No (unless sync is enabled) |
| Works with env vars | Yes | Only with sync to Secret |
| Fetch time | Background, every `refreshInterval` | At Pod start (mount), optional rotation polling |
| Vault down at Pod start | Pod still starts (Secret exists) | Pod **fails to start** (mount fails) |
| Good for | most apps, GitOps | strict "never store in etcd" rules |

### How it works internally
```
 Pod with volume: csi driver secrets-store.csi.k8s.io, volumeAttributes.secretProviderClass=app-secrets
   |
 kubelet --> Secrets Store CSI node plugin (DaemonSet) --NodePublishVolume-->
   --> provider plugin (e.g. OCI) via gRPC --> fetch secret from vault using the Pod's identity
   --> write files into a tmpfs --> mount into the Pod
```

### Real production scenario
A banking app on OKE must not have its card-encryption key stored in etcd at all (security rule). You use the Secrets Store CSI Driver with the OCI provider. The key exists only in OCI Vault and, at runtime, as a tmpfs file inside the Pod.

### Lab — concept and manifests (not run locally)
This needs a real vault and provider. The install pattern is:
```bash
helm repo add secrets-store-csi-driver https://kubernetes-sigs.github.io/secrets-store-csi-driver/charts
helm install csi-secrets-store secrets-store-csi-driver/secrets-store-csi-driver -n kube-system \
  --set syncSecret.enabled=true --set enableSecretRotation=true
# then install the provider plugin for your vault (OCI, Azure, AWS, GCP or HashiCorp Vault)
```
Example manifests (OCI provider — check parameter names in the provider's docs):
```yaml
apiVersion: secrets-store.csi.x-k8s.io/v1
kind: SecretProviderClass
metadata:
  name: card-keys
  namespace: payments
spec:
  provider: oci                       # provider plugin name
  parameters:                         # provider-specific
    secrets: |
      - name: card-encryption-key     # secret name in OCI Vault
        fileName: card.key            # file name in the Pod
    vaultId: ocid1.vault.oc1.me-jeddah-1.<unique-id>
    authType: workload                # OKE Workload Identity
---
apiVersion: v1
kind: Pod
metadata: {name: payments-api, namespace: payments}
spec:
  serviceAccountName: payments-api    # identity used to read the vault
  containers:
    - name: api
      image: payments-api:3.1.0
      volumeMounts:
        - name: keys
          mountPath: /mnt/secrets
          readOnly: true
  volumes:
    - name: keys
      csi:
        driver: secrets-store.csi.k8s.io
        readOnly: true
        volumeAttributes:
          secretProviderClass: card-keys
```

### Troubleshooting Secrets Store CSI
| Symptom | Cause | Fix |
|---|---|---|
| Pod `ContainerCreating`, `failed to mount secrets store objects` | Auth failed or secret name wrong | `kubectl describe pod`; provider pod logs; IAM policy for the workload |
| Pod starts in one cluster, not another | Provider plugin DaemonSet missing on that node pool | Check DaemonSet tolerations |
| Rotation does not reach the app | Rotation disabled, or app caches the value | `enableSecretRotation=true`; app must re-read the file |

### Lab vs Production
| Lab | Production |
|---|---|
| Only manifests | Driver + provider installed and upgraded by the platform team |
| — | Decide per app: ESO (simpler, resilient) or CSI (no etcd copy) |

### Check yourself
1. Which tool avoids storing the secret in etcd?
<details><summary>Answer</summary>Secrets Store CSI Driver (with secret sync disabled).</details>

2. The vault is down for 10 minutes. Which approach still lets new Pods start?
<details><summary>Answer</summary>ESO: the Kubernetes Secret already exists. With the CSI driver, the mount needs the vault at Pod start.</details>

---
# Interview and real-world questions

1. **"Our Secrets are base64-encoded, so they are safe in Git." Respond.**
<details><summary>Answer</summary>Base64 is an encoding, not encryption — anyone can decode it. Secrets must not be in Git. Use External Secrets (references only in Git), Sealed Secrets or SOPS (encrypted values), and rotate anything already committed.</details>

2. **How do you rotate a database password in Kubernetes with zero downtime?**
<details><summary>Answer</summary>Create a second DB user/password (or use dual-password support), update the vault, let ESO sync the Secret, roll out the Deployment so Pods pick up the new value (hash/annotation triggers), verify, then remove the old credential. Apps reading the secret from a file and reloading can avoid restarts.</details>

3. **Why did my ConfigMap change not take effect?**
<details><summary>Answer</summary>Env vars are set only at container start; <code>subPath</code> mounts never update; the app may not re-read files. Fix with a rollout (hash suffix, <code>kubectl rollout restart</code>, Reloader) or app reload.</details>

4. **How is a Kubernetes Secret protected end-to-end?**
<details><summary>Answer</summary>TLS to the API server; RBAC on get/list/watch; encryption at rest in etcd (KMS); kubelet only receives Secrets for Pods on its node; tmpfs on the node; Pod Security limits who can mount them; audit logs; source of truth in a vault with rotation.</details>

5. **ESO or Secrets Store CSI for a new platform?**
<details><summary>Answer</summary>Default to ESO: simpler, works with env vars, Pods start even if the vault is briefly down, fits GitOps. Use the CSI driver where policy forbids secrets in etcd, accepting the start-time dependency on the vault.</details>

6. **What does OKE use for Secret encryption at rest?**
<details><summary>Answer</summary>Envelope encryption with a master encryption key that you choose from OCI Vault when creating/updating the cluster; an IAM policy must allow the cluster to use the key.</details>

---
# Practical Assignment — "Move the payments service to proper config"

**Scenario:** The `payments` service has its config and password hard-coded in the Deployment:
```yaml
env:
  - {name: DB_PASSWORD, value: "Hardcoded-Pass-1"}
  - {name: LOG_LEVEL, value: "info"}
  - {name: TIMEOUT_MS, value: "3000"}
```
Fix it in namespace `lab-06`:
1. Put `LOG_LEVEL` and `TIMEOUT_MS` into a kustomize-generated ConfigMap `payments-config` (hash suffix, immutable).
2. Get `DB_PASSWORD` from the ESO fake store via an `ExternalSecret` named `payments-db` (vault key `/prod/payments/db`, value `Vault-Pay-Pass-1`), target Secret `payments-db`.
3. The Deployment `payments` (busybox, prints its env) uses `envFrom` for the ConfigMap and `secretKeyRef` for the password. Add `POD_NAME` and `NODE_NAME` via the Downward API.
4. Change `TIMEOUT_MS` to `5000` and prove that a new rollout happens.

**Success criteria**
- `kubectl get externalsecret payments-db` is `SecretSynced`.
- `kubectl logs deploy/payments` shows `DB_PASSWORD=Vault-Pay-Pass-1`, `TIMEOUT_MS=5000`, `POD_NAME=...`.
- `kubectl rollout history deploy/payments` shows 2 revisions.

**Hints:** reuse the files from Topic 5 and Topic 7; put everything in one kustomize folder `labs/06/payments`.

<details><summary>Full solution</summary>

`labs/06/payments/kustomization.yaml`
```yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
namespace: lab-06
resources: [deployment.yaml, external-secret.yaml]
configMapGenerator:
  - name: payments-config
    literals: [LOG_LEVEL=info, TIMEOUT_MS=3000]
    options: {immutable: true}
```
`labs/06/payments/external-secret.yaml`
```yaml
apiVersion: external-secrets.io/v1
kind: SecretStore
metadata: {name: payments-fake-vault}
spec:
  provider:
    fake:
      data:
        - {key: /prod/payments/db, value: "Vault-Pay-Pass-1", version: v1}
---
apiVersion: external-secrets.io/v1
kind: ExternalSecret
metadata: {name: payments-db}
spec:
  refreshInterval: 1m
  secretStoreRef: {name: payments-fake-vault, kind: SecretStore}
  target: {name: payments-db, creationPolicy: Owner}
  data:
    - secretKey: DB_PASSWORD
      remoteRef: {key: /prod/payments/db, version: v1}
```
`labs/06/payments/deployment.yaml`
```yaml
apiVersion: apps/v1
kind: Deployment
metadata: {name: payments}
spec:
  replicas: 1
  selector: {matchLabels: {app: payments}}
  template:
    metadata: {labels: {app: payments}}
    spec:
      containers:
        - name: app
          image: busybox:1.36
          command: ["sh", "-c", "env | sort; sleep 3600"]
          envFrom:
            - configMapRef: {name: payments-config}
          env:
            - name: DB_PASSWORD
              valueFrom: {secretKeyRef: {name: payments-db, key: DB_PASSWORD}}
            - name: POD_NAME
              valueFrom: {fieldRef: {fieldPath: metadata.name}}
            - name: NODE_NAME
              valueFrom: {fieldRef: {fieldPath: spec.nodeName}}
```
```bash
kubectl apply -k labs/06/payments
kubectl rollout status deploy/payments
# change TIMEOUT_MS=3000 -> 5000 in kustomization.yaml, then:
kubectl apply -k labs/06/payments
kubectl rollout history deploy/payments
kubectl logs deploy/payments | grep -E 'DB_PASSWORD|TIMEOUT_MS|POD_NAME'
```
(If the first Pod starts before ESO creates the Secret, it shows `CreateContainerConfigError` for a few seconds, then starts — kubelet retries.)
</details>

---
# "Break it" challenge

This Pod will not start, and even when it starts, it does not get the config the developer expected. Find **all** problems.

In [ ]:
%%writefile labs/06/break-it.yaml
apiVersion: v1
kind: Secret
metadata:
  name: api-creds
  namespace: lab-06
type: Opaque
data:
  API_TOKEN: my-plain-token
---
apiVersion: v1
kind: ConfigMap
metadata:
  name: api-config
  namespace: default
data:
  api.timeout: "30"
  RETRIES: 3
---
apiVersion: v1
kind: Pod
metadata:
  name: api-client
  namespace: lab-06
spec:
  containers:
    - name: client
      image: busybox:1.36
      command: ["sh", "-c", "env; sleep 3600"]
      envFrom:
        - configMapRef:
            name: api-config
      env:
        - name: API_TOKEN
          valueFrom:
            secretKeyRef:
              name: api-creds
              key: api_token

In [ ]:
%%bash
kubectl apply -f labs/06/break-it.yaml 2>&1
sleep 5
kubectl get pod api-client 2>&1

<details><summary>Answer</summary>

1. **Secret `data` must be base64.** `my-plain-token` is not valid base64 → apply fails (`illegal base64 data`). Use `stringData:` for plain text.
2. **`RETRIES: 3` is unquoted** → it is a number, and ConfigMap `data` values must be strings → apply fails. Quote it: `"3"`.
3. **ConfigMap is in namespace `default`**, the Pod is in `lab-06` → `CreateContainerConfigError: configmap "api-config" not found`. Pods can only use ConfigMaps/Secrets from their **own** namespace.
4. **`key: api_token`** but the Secret key is `API_TOKEN` → keys are case-sensitive → `couldn't find key api_token`.
5. **`api.timeout` will be skipped** by `envFrom` (a dot is not a valid env var name). Rename to `API_TIMEOUT` or mount as a file.

Fixed:
```yaml
apiVersion: v1
kind: Secret
metadata: {name: api-creds, namespace: lab-06}
type: Opaque
stringData: {API_TOKEN: my-plain-token}
---
apiVersion: v1
kind: ConfigMap
metadata: {name: api-config, namespace: lab-06}
data: {API_TIMEOUT: "30", RETRIES: "3"}
---
# Pod: secretKeyRef key: API_TOKEN
```
</details>

---
# Cleanup

Deletes the lab namespace and resets your default namespace. ESO stays installed (it is useful later); uncomment the last line to remove it too.

In [ ]:
%%bash
ctx=$(kubectl config current-context); [ "$ctx" = "kind-learn" ] || { echo "STOP: context is $ctx"; exit 1; }
kubectl delete namespace lab-06 --wait=true --timeout=180s
rm -f labs/06/tls.crt labs/06/tls.key
kubectl config set-context --current --namespace=default
# helm uninstall external-secrets -n external-secrets && kubectl delete namespace external-secrets
echo "Cleanup done"

---
**Next notebook:** [07-Scheduling-and-Resources.ipynb](07-Scheduling-and-Resources.ipynb) — requests and limits, QoS, affinity, taints, PDBs and autoscaling.